# CU28 mixed_context - Feature Engineering Audit

Notebook narrativo de auditoria para el scope `mixed_context`.


## Objetivo

Mostrar de forma explicita como se estructura el dataset modelable, que features entran en cada etapa y como se controla el leakage.


## Alcance

Este analisis describe la ruta oficial reproducible `mixed_context`. Las senales externas se tratan como contexto/proxy. Las variables internas de planta siguen siendo sinteticas salvo carga posterior de cliente.


## Inputs

            - `data/processed/baseline/feature_engineering_modeling__mixed_context.csv`
- `data/processed/baseline/modeling_metadata__mixed_context.json`
- `data/processed/baseline/feature_contract__mixed_context.csv`
- `data/processed/baseline/feature_roles_metadata__mixed_context.json`
- `docs/input_contract.md`
- `docs/feature_engineering.md`
- `docs/leakage_policy.md`


## Outputs esperados

            - `reports/tables/eda/feature_inventory__mixed_context.csv`
- `reports/tables/eda/leakage_audit__mixed_context.csv`
- `reports/tables/eda/feature_sets_allowed__mixed_context.csv`
- `reports/figures/eda/features_by_family__mixed_context.png`
- `reports/figures/eda/features_by_origin__mixed_context.png`
- `reports/figures/eda/feature_missing_values__mixed_context.png`
- `reports/figures/eda/feature_correlation__mixed_context.png`
- `reports/figures/eda/feature_target_correlation__mixed_context.png`


## Limitaciones

Este notebook documenta evidencia reproducible del pipeline oficial, pero no sustituye la revision de codigo, la auditoria de datos de origen ni una certificacion operacional de planta.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.reproducibility.notebook_support import (
    detect_temporal_columns,
    ensure_eda_dirs,
    execution_metadata,
    first_valid_temporal_range,
    load_source_manifests,
    load_tabular_file,
    parse_markdown_table,
    print_frame,
    print_series,
    project_root,
    read_json,
    relative_to_root,
    save_figure,
    save_table,
    sha256_file,
)

from src.reproducibility.mixed_context import quantity_feature_columns, trigger_feature_columns


In [ ]:
NOTEBOOK_NAME = "04_feature_engineering_audit.ipynb"
PROJECT_ROOT = project_root()
SCOPE = globals().get("scope", "mixed_context")
REPORT_DIRS = ensure_eda_dirs()
META = execution_metadata(SCOPE)
FIGURES = []
TABLES = []
print(json.dumps(META, indent=2))


## Carga de datos

Las siguientes celdas cargan los artefactos de entrada y muestran verificaciones intermedias antes de producir tablas y graficas.


In [ ]:
modeling_path = PROJECT_ROOT / "data/processed/baseline/feature_engineering_modeling__mixed_context.csv"
metadata_path = PROJECT_ROOT / "data/processed/baseline/modeling_metadata__mixed_context.json"
feature_contract_path = PROJECT_ROOT / "data/processed/baseline/feature_contract__mixed_context.csv"
feature_roles_path = PROJECT_ROOT / "data/processed/baseline/feature_roles_metadata__mixed_context.json"
input_contract_path = PROJECT_ROOT / "docs" / "input_contract.md"
modeling_df = pd.read_csv(modeling_path)
metadata = read_json(metadata_path)
feature_contract = pd.read_csv(feature_contract_path)
feature_roles = read_json(feature_roles_path)
input_contract_excerpt = input_contract_path.read_text(encoding="utf-8").splitlines()[:20]
print(modeling_df.shape)
print_frame("Feature contract preview", feature_contract.head(12))


In [ ]:
print("Input contract excerpt")
for line in input_contract_excerpt:
    print(line)
print(json.dumps({"recommended_feature_set": feature_roles.get("recommended_feature_set")}, indent=2))


## Inspeccion inicial

A continuacion se separan targets, outputs, inputs permitidos y columnas excluidas para dejar trazabilidad explicita de la fase de feature engineering.


In [ ]:
shape_summary = pd.DataFrame([{"rows": len(modeling_df), "columns": len(modeling_df.columns)}])
column_summary = pd.DataFrame({"feature_name": modeling_df.columns})
print_frame("Modeling dataset shape", shape_summary)
print_frame("Modeling columns", column_summary, rows=30)


In [ ]:
targets_df = feature_contract[feature_contract["output_role"] == "predictive_target"].copy()
outputs_df = feature_contract[feature_contract["output_role"] == "decision_output"].copy()
inputs_df = feature_contract[feature_contract["allowed_model_input"] == "yes"].copy()
excluded_df = feature_contract[feature_contract["allowed_model_input"] == "no"].copy()
print_frame("Targets", targets_df)
print_frame("Decision outputs", outputs_df)
print_frame("Allowed model inputs", inputs_df.head(20))


In [ ]:
lag_features = feature_contract[feature_contract["feature_name"].str.contains("_lag_", na=False)].copy()
rolling_features = feature_contract[feature_contract["feature_name"].str.contains("_roll_mean_", na=False)].copy()
calendar_features = feature_contract[feature_contract["feature_origin"] == "calendar_derived"].copy()
profile_features = feature_contract[
    feature_contract["feature_name"].str.startswith(
        ("product_family__", "recipe_profile__", "shelf_life_class__", "manufacturing_context_profile__"),
        na=False,
    )
].copy()
print_frame("Lag features", lag_features.head(20))
print_frame("Rolling features", rolling_features.head(20))
print_frame("Calendar features", calendar_features.head(20))
print_frame("Profile features", profile_features.head(20))


In [ ]:
missing_summary = modeling_df.isna().mean().reset_index()
missing_summary.columns = ["feature_name", "missing_pct"]
missing_summary = missing_summary.sort_values("missing_pct", ascending=False)
constant_features = pd.DataFrame(
    {"feature_name": [column for column in modeling_df.columns if modeling_df[column].nunique(dropna=False) <= 1]}
)
print_frame("Missing summary", missing_summary.head(20))
print_frame("Constant features", constant_features.head(20))


In [ ]:
feature_inventory = feature_contract[[
    "feature_name",
    "feature_origin",
    "feature_type",
    "system_layer",
    "temporal_relation_to_target",
    "output_role",
    "allowed_model_input",
    "leakage_risk",
]].copy()
print_frame("Feature inventory", feature_inventory.head(20))
display(feature_inventory.head(20))


## Auditoria de leakage

La politica oficial excluye salidas downstream y señales de decision final como inputs indebidos en la fase upstream y en la fase trigger.


In [ ]:
prohibited_features = [
    "order_quantity_tons",
    "quantity_optimizer_recommendation_tons",
    "quantity_optimizer_target_tons",
    "excess_tons",
    "stockout_tons",
    "purchase_trigger_flag",
    "purchase_trigger_proba",
]
trigger_allowed = set(trigger_feature_columns(modeling_df))
quantity_allowed = set(quantity_feature_columns(modeling_df))
upstream_allowed = set(feature_roles.get("official_extended_inputs", []))
leakage_rows = []
for feature_name in prohibited_features:
    leakage_rows.append(
        {
            "feature_name": feature_name,
            "present_in_dataset": feature_name in modeling_df.columns,
            "in_upstream": feature_name in upstream_allowed,
            "in_trigger": feature_name in trigger_allowed,
            "in_quantity_optimizer": feature_name in quantity_allowed,
            "status": "fail" if (feature_name in upstream_allowed or feature_name in trigger_allowed) else "pass",
        }
    )
leakage_audit_df = pd.DataFrame(leakage_rows)
print_frame("Leakage audit", leakage_audit_df, rows=20)
display(leakage_audit_df)


In [ ]:
feature_sets_allowed = pd.DataFrame(
    [
        {"stage": "upstream_predictor", "feature_name": feature_name}
        for feature_name in sorted(upstream_allowed)
    ]
    + [
        {"stage": "purchase_trigger", "feature_name": feature_name}
        for feature_name in sorted(trigger_allowed)
    ]
    + [
        {"stage": "quantity_optimizer", "feature_name": feature_name}
        for feature_name in sorted(quantity_allowed)
    ]
)
print_frame("Allowed feature sets by stage", feature_sets_allowed.head(30))
display(feature_sets_allowed.head(30))


In [ ]:
origin_counts = feature_contract["feature_origin"].value_counts().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(origin_counts.index, origin_counts.values, color="#355c7d")
ax.set_title("Features by origin")
ax.set_ylabel("count")
ax.tick_params(axis="x", rotation=45)
FIGURES.append(save_figure(fig, "features_by_origin__mixed_context.png"))
plt.close(fig)


In [ ]:
family_counts = feature_contract["feature_type"].value_counts().sort_values(ascending=False).head(12)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(family_counts.index, family_counts.values, color="#6c5b7b")
ax.set_title("Features by family")
ax.set_ylabel("count")
ax.tick_params(axis="x", rotation=45)
FIGURES.append(save_figure(fig, "features_by_family__mixed_context.png"))
plt.close(fig)


In [ ]:
missing_plot = missing_summary.head(20).copy()
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(missing_plot["feature_name"], missing_plot["missing_pct"], color="#f67280")
ax.set_title("Top missing values after feature engineering")
ax.set_ylabel("missing_pct")
ax.tick_params(axis="x", rotation=75)
FIGURES.append(save_figure(fig, "feature_missing_values__mixed_context.png"))
plt.close(fig)


In [ ]:
numeric_candidates = modeling_df.select_dtypes(include=["number"]).copy()
correlation_columns = ["synthetic_procurement_need", "purchase_trigger_label", "demand_index", "supply_index", "current_inventory_tons", "expected_requirement_tons"]
correlation_columns = [column for column in correlation_columns if column in numeric_candidates.columns]
corr = numeric_candidates[correlation_columns].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(7, 5))
image = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.index)
ax.set_title("Key feature correlation")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
FIGURES.append(save_figure(fig, "feature_correlation__mixed_context.png"))
plt.close(fig)


In [ ]:
target_corr = numeric_candidates.corr(numeric_only=True)["synthetic_procurement_need"].dropna().sort_values(key=lambda s: s.abs(), ascending=False).head(15)
target_corr_df = target_corr.reset_index()
target_corr_df.columns = ["feature_name", "correlation_with_target"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(target_corr_df["feature_name"], target_corr_df["correlation_with_target"], color="#2a9d8f")
ax.set_title("Correlation with synthetic_procurement_need")
ax.tick_params(axis="x", rotation=75)
FIGURES.append(save_figure(fig, "feature_target_correlation__mixed_context.png"))
plt.close(fig)
print_frame("Correlation with synthetic_procurement_need", target_corr_df, rows=20)


## Interpretacion

La auditoria confirma que las salidas downstream no entran como inputs indebidos en upstream ni trigger. `purchase_trigger_label` y su probabilidad solo aparecen como entradas legitimas del quantity optimizer.


In [ ]:
TABLES.append(save_table(feature_inventory, "feature_inventory__mixed_context.csv"))
TABLES.append(save_table(leakage_audit_df, "leakage_audit__mixed_context.csv"))
TABLES.append(save_table(feature_sets_allowed, "feature_sets_allowed__mixed_context.csv"))
RESULT = {
    "notebook": NOTEBOOK_NAME,
    "tables": TABLES,
    "figures": FIGURES,
    "findings": [
        "Feature engineering is traceable through the feature contract and roles metadata.",
        "Lagged, rolling and profile-derived variables are visible as separate families.",
        "Downstream decision outputs are excluded from upstream and trigger inputs.",
    ],
    "limitations": [
        "The notebook audits the exported feature contract; it does not replace code review of the full feature pipeline.",
    ],
}
print(json.dumps(RESULT, indent=2))


## Limitaciones

El notebook audita los artefactos exportados de feature engineering. No sustituye la lectura del codigo completo ni la inspeccion manual de todas las transformaciones intermedias.


## Concluson final

La fase de feature engineering queda defendible cuando los conjuntos permitidos por etapa son visibles, las exclusiones por leakage son explicitas y el dataset modelable puede reconstruirse a partir de artefactos versionados.
